```mermaid
sequenceDiagram
 %% Define actors and participants (eg systems and databases)
    participant AD as AstroDrive
    Actor AST as Astronomer
    participant HOPS
    participant SIMBAD
%%  participant SIMBAD @{ "type" : "database" }

%% add message numbering
    autonumber

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% Define Process flow
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical Retrieve image files from AstroDrive
    
        note over AD: Automatic upload of OSO images
        AST-->>AD:Log onto server
        AST-->>AD: Judgement Call.<br/>Initial image quality check 
        note over AST, AD: 1)Is the target star well<br/> away from saturation?<br/>2)Are star images approx. circular?<br/>3)is SNR>10?<br/>4)Are there any artefacts<br/> in the image?
        AD-->>AST:Download dataset to local storage
        
    end

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% Prepare data for HOPS processing
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical Prepare data for HOPS processing
    
        AST->>AST:Organise image files into one folder
        AST->>AST:Judegement Call.<br/>Complete input data checklist
        note over AST:From OU Manual<br/>1)Do calibration files have the<br/> same binning as science data?<br/>2)Were flat fields taken with<br/> same filter as science frames?<br/>How checked ?<br/> Remove flat-field frames<br/> with a different filter<br/>3) Remove any dark frames
        
        note over AST:From HOPS Manual<br/>1)Use the same camera temperature,<br/> binning and subframe as the science frames<br/>2) Obtain at least five bias frames<br/> (zero exposure, using a cover),<br/>and check that there is no<br/> external light contaminating them.<br/> 3) obtain at least five dark frames<br/> (same exposure time as the science frames,<br/> using a cover)<br/>and check that there is no<br/> external light contaminating them.<br/> 4)Obtain at least five flat frames<br/> (pointing to a uniformly illuminated surface,<br/> with the counts at 2/3 of <br/>the full well-depth of your camera),<br/> if you are using the sky,<br/> check that stars are not visible in your frames.<br/>5 Do not apply any pre-processing<br/> (for example do not create master frames)<br/> HOPS will create the master frames <br/>on the fly and use them appropriately.

        note over AST: Judgement Call:Can scripts<br/> automate some of these checks
        
    end

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% HOPS Step 0. Configure HOPS profile
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%    
    critical HOPS Step 0. Configure HOPS profile
    
        AST-->>HOPS:Set up HOPS profile<br/>MY PROFILE on main menu
        note over AST,HOPS:Set up keywords for extract strings
        note over AST,HOPS: Would the ability to switch keywords<br/>help an Astronomer who<br/> uses multiple telescopes?
       
    end

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% HOPS Step 1. Select Data & Target
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical HOPS Step 1. Select Data & Target
    
        AST-->>HOPS:Select/Change Directory
        AST-->>HOPS:Set "Name identifiers" for files
        AST-->>HOPS:Set "Header information" 
        
        note over AST,HOPS: Exposure time key = EXPTIME<br/> for Spies and OSO data
        note over AST,HOPS: Observation date key<br/> = GPSSTART for OSO data or<br/> TIME-OBS for Spies
        note over AST,HOPS: Is there an override to take<br/> observation time from DATE-OBS<br/> if TIME-OBS does not exist in FITS?
        note over AST,HOPS:Time-stamp = exposure start<br/> Filter set as appropriate
        
        AST-->>HOPS:Ensure Location is correct for telescope
        note over AST,HOPS:For OSO PIRATE Location<br/> +28:17:57.984 343:29:22.992
        
        alt Telescope location options
            HOPS->>HOPS:Retrieve location from image header
        else
            HOPS->>HOPS:Retrieve from MY PROFILE
        else
            AST-->>HOPS:Manually enter LAT/LONG
        end

        
        AST-->>HOPS:Ensure Target is correct for<br/> objective star
        alt Target location options
            HOPS->>HOPS:Retrieve location from image header
            note over HOPS:Test data HAT-P-16<br/> target location in header not accurate.
        else
            HOPS-->>SIMBAD:Retrieve location from SIMBAD
        else
            AST-->>HOPS:Manually enter RA/DEC
        end
        note over AST,HOPS:add to flow<br/>  Advanced Setting and Observer information
        
    end
    
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% HOPS Step 2. Run Reduction
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical HOPS Step 2. Run Reduction

        AST-->>HOPS:Run data reduction step
        HOPS->>HOPS: Creates new folder REDUCED_DATA<br/> in selected directory
        note over AST,HOPS:if rerun output files are overriden 
    
    end

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% HOPS Step 3. Inspect Frames
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical HOPS Step 3. Inspect Frames

        AST-->>HOPS: DQ check that allows for manual filtering out of <br/> faulty reduced science frames
        
        alt OU Manual
            AST-->>HOPS: Exclude 1st frame by default. 
        else HOPS Manual
            AST-->>HOPS: Judgement Call.<br/> Exclude 1st frame if poor quality
        end
    
        AST-->>HOPS: Judgement Call.<br/> Assess SKY graph to identify frames to exclude
        note over AST,HOPS: What is criteria to exclude frames from SKY graph?<br/>Remove frames which are off-trend from rest of pop.?<br/> How do we quantify this?<br/> How do exclusion improve the final results
        
        AST-->>HOPS: Judgement Call.<br/> Assess PSF graph to identify frames to exclude
        note over AST,HOPS:HOPS Manual-exclude frames where<br/> HWHM>= 2.5
        note over AST,HOPS: How are good frames identified?
        note over AST,HOPS: try with 0 exclusions then increase 
    
    end
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% HOPS Step 4. Run alignment
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical HOPS Step 4. Run alignment

        AST-->>HOPS: exclude images if "Stars not found close to previous positions"

        note over AST,HOPS: Are target location in image headers accurate enough?
    
    end
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% HOPS Step 5. Photometry
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical HOPS Step 5. Photometry

        alt Image from Open University OSO
            AST->>AST: Do NOT plate solve    
        else Image NOT from OU
            AST-->>HOPS: Plate solve image 
            HOPS-->>SIMBAD:Check SIMBAD to solve
               
            alt Plate solve successful
                SIMBAD-->>HOPS: Return result and highlight target on image
            else Plate solve unsuccessful
                SIMBAD-->>HOPS: Return failed message
                note over SIMBAD,HOPS: Plate solution failed.<br/> Please check your internet connection<br/> and the "Star FWHM in arcseonds" value,<br/> under the "Advanced settings".
                note over SIMBAD,HOPS: Target coordinates (HAT-P-16)<br/> fail to plate solve
            end
        end

        AST-->>HOPS: Select Target star
        note over AST,HOPS: Max Counts value: How is this used to determine<br/> saturation point limit?

        AST-->>HOPS: Judgement Call.<br/>Select comparison stars for checking
        AST-->>HOPS: Judgement Call.<br/>Check comparison starts V-Mag and Gaia BP-RP (Colour)<br/> similar to target values
        AST-->>SIMBAD: Judgement Call.Use SIMAB to find comparison stars and information

        AST-->>HOPS: Judgement Call.<br/>Select comparison stars to use for photometry
        note over AST,HOPS: When removing stars, make sure the right ones are removed.
        note over AST,HOPS: HOPS Manual - Advanced options for PSF: Size change and asymmetries

        AST-->>HOPS: Judgement Call.<br/>Ensure aperture sizes encompass star(s) appropriately.

        AST-->>HOPS: Run Photometry

        AST-->>HOPS: Judgement Call.<br/>Select Light curves that best match Target star
    
    end

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%% HOPS Step 6. Exoplanet Fitting
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    critical HOPS Step 6. Exoplanet Fitting

        AST-->>HOPS: Enter exoplanet parameters
        alt Exoplanet parameters found in HOPS Catalogues
            HOPS-->>HOPS: Retrieve exoplanet parms from database
            note over HOPS: What database holds parms
        else Exoplanet parameters entered manually
            AST-->>HOPS: Add exoplanet parameters into HOPS
        end

        AST-->>HOPS: Select light curve file - Apeture_x

        AST-->>HOPS: Judgement Call.<br/> Set detrending parameter (airmass, linear or quadratic)

        AST-->>HOPS: RUN TEST exoplanet fitting

        AST-->>AST: Judgement Call. Evaluate test results
        note over AST: How is a "good test" measured?<br/> Does transit depth match expected model?
        note over AST: Will different comparison stars<br/> improve results?
        note over AST: Are STD and AutoCorrelation<br/> of residuals minimised?

        alt RUN TEST step successful
            AST-->>HOPS: Set MCMC parameters(total steps and burn in)
        else RUN TEST step unsuccessful
            AST-->>HOPS: Adjust photometric parameters, comparison stars etc
        end

        AST-->>HOPS: Run Fitting for final exoplanet analysis.
        AST->>AST: Create output files<br/>log.yaml, all_frames.pickle, all_stars.pickle

        AST-->>HOPS: Export results to text files and ExoClock database.
        


    end
```